# DAG-SA v2 — incremental study of six proposed improvements

The revised manuscript reports that DAG-SA is competitive with but does not outperform standard baselines, and diagnoses the cause as the **selection signal**: topology choice is driven by accuracy on ~30 validation trials over ~10^11 candidates, and the stacking operator is scored in-sample on that same split.

This notebook tests six changes aimed at that diagnosis. Each variant differs from the published method in **exactly one** respect and runs on **identical splits and seeds**, so every comparison is paired and McNemar's exact test applies.

| Variant | Change |
|---|---|
| `V0_published` | the published method (reference) |
| `V1_oof_objective` | out-of-fold objective + cross-validated stacking |
| `V2_regularised_selection` | complexity penalty + one-standard-error rule |
| `V3_topk_average` | average of the 5 best topologies instead of the argmax |
| `V4_enriched_pool` | tangent-space, FBCSP **and the exact B5 baseline** as pool members |
| `V7_strong_members` | the B4 (EEGNet) and B5 baselines themselves as pool members (needs torch) |
| `V5_diversity` | diversity-regularised objective |
| `V6_shrinkage_csp` | OAS shrinkage for the CSP covariance |
| `VALL_all_six` | everything at once |

**Before you start:** decide now that you will report the outcome whichever way it goes. Running variants until one wins and reporting only that one is exactly the practice the current revision exists to correct.

## 0. Clone the repository
The v2 code lives in the `v2/` folder of the **`v2-improvements`** branch;
`main` holds the published pipeline cited in the manuscript and does not contain it.

In [ ]:
REPO_URL = 'https://github.com/yazanjer/DAG-Ensembles-EEG.git'   # private repo: https://<TOKEN>@github.com/...
BRANCH   = 'v2-improvements'   # the v2/ folder lives on this branch, not on main

import os, sys, glob, subprocess
CLONE_DIR = '/content/eeg_dagsa_repo'
if not os.path.exists(CLONE_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, CLONE_DIR], check=True)
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', CLONE_DIR, 'pull', 'origin', BRANCH])

hits = glob.glob(CLONE_DIR + '/**/run_v2.py', recursive=True)
assert hits, ('run_v2.py not found - are you on the ' + BRANCH + ' branch? '
              'The v2 code is in the v2/ folder of that branch.')
CODE_DIR = os.path.dirname(hits[0])
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print('branch :', BRANCH)
print('code dir:', CODE_DIR)

## 1. Dependencies
`torch` is only needed if you also want the EEGNet baseline; the v2 study does not use it.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install',
                'numpy', 'scipy', 'scikit-learn', 'pandas', 'matplotlib',
                'pyyaml', 'mne', 'pyriemann'])
# torch is needed only by V7_strong_members (EEGNet as a pool member);
# it is pre-installed on most Colab runtimes.
try:
    import torch
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch'])
import numpy, scipy, sklearn, mne, pyriemann
print('deps ready:', numpy.__version__, sklearn.__version__, pyriemann.__version__)
try:
    import torch; print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())
except Exception:
    print('torch NOT available - V7_strong_members will skip its EEGNet members')

## 2. Mount Drive and set the project root
Expected layout on Drive:
```
MyDrive/EEG_DAGSA/dataset/BCICIV_calib_ds1a.mat ...    (Dataset 1)
MyDrive/EEG_DAGSA/dataset_2a/A01T.mat ...              (Dataset 2a)
```
Results are written to `MyDrive/EEG_DAGSA/results/` so they survive a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/EEG_DAGSA'
os.environ['EEG_DAGSA_ROOT'] = PROJECT_ROOT
DS1_DIR = PROJECT_ROOT + '/dataset'
DS2A_DIR = PROJECT_ROOT + '/dataset_2a'
print(sorted(os.listdir(DS1_DIR))[:4])

## 3. Wiring check (about a minute)
A deliberately tiny pool and 5 search iterations — this only proves every code path runs, and the numbers mean nothing.

In [ ]:
import run_v2
df = run_v2.run_campaign(dataset='ds1', subjects=['a'], seeds=[42],
                         experiment='smoke_v2', dataset_dir=DS1_DIR,
                         config_path='config_smoke.yaml', tiny=True)
df[['variant', 'accuracy', 'n_pool', 'seconds']]

## 4. The real campaign — Dataset 1

All eight variants, four subjects, eight seeds. Measured cost per (subject, seed) unit on a 4-core CPU: roughly 8–12 minutes for the whole set of variants, dominated by the out-of-fold matrices (`V1` and `VALL`) which refit the entire feature pipeline once per inner fold.

| Scope | Units | Rough wall-clock (CPU) |
|---|---|---|
| all 8 variants, 4 subjects × 8 seeds | 32 | 4–6 h |
| all 8 variants, 4 subjects × 3 seeds | 12 | 1.5–2.5 h |
| cheap variants only (`V0 V2 V3 V4 V5 V6`) | 32 | 1–1.5 h |

The run writes after **every** unit, so a disconnect costs at most one unit — restart the cell and it will overwrite cleanly. If you are short of time, run the cheap variants first: they tell you whether the selection-rule changes matter before you pay for the out-of-fold ones.

In [ ]:
import importlib, run_v2, variants
importlib.reload(variants); importlib.reload(run_v2)

df1 = run_v2.run_campaign(
    dataset='ds1',
    subjects=['a', 'b', 'f', 'g'],
    seeds=[42, 43, 44, 45, 46, 47, 48, 49],
    variant_names=None,          # None = all eight; or e.g. ['V0_published','V1_oof_objective']
    experiment='ds1_v2',
    dataset_dir=DS1_DIR,
    verbose=True)
df1.groupby('variant')['accuracy'].mean().sort_values(ascending=False)

## 5. Read the result
`Δ vs V0` is the mean paired difference; `W/T/L` counts a win or loss only where McNemar reaches p < 0.05.

In [ ]:
import analyse_v2
s = analyse_v2.report(__import__('pathlib').Path(PROJECT_ROOT) / 'results' / 'ds1_v2')

## 6. Dataset 2a — the real test of the enriched pool

This is where EEGNet beats DAG-SA by 6.4 points, so it is the dataset where embedding the strong baselines can actually change the answer.

* `V4_enriched_pool` — tangent-space and FBCSP views **plus the exact B5 baseline** (unfiltered epochs, logistic-regression head) as selectable members.
* `V7_strong_members` — adds **EEGNet itself** as a pool member, so a committee can be a committee of strong baselines.

Cost: nine subjects x eight seeds = 72 units. V7 trains one EEGNet per unit, so budget roughly 2-4 h on a CPU runtime and well under that on GPU. Results are written after every unit.

**Read the result carefully.** If DAG-SA only becomes competitive because the pool now contains EEGNet, the claim changes from "annealed search over a CSP pool" to "ensemble selection over strong heterogeneous members". That is a legitimate paper, but a different one — decide which you are writing before the numbers arrive.

In [ ]:
df2 = run_v2.run_campaign(
    dataset='ds2a',
    subjects=[1, 2, 3, 4, 5, 6, 7, 8, 9],
    seeds=[42, 43, 44, 45, 46, 47, 48, 49],
    variant_names=['V0_published', 'V4_enriched_pool', 'V7_strong_members'],
    experiment='ds2a_strong',
    dataset_dir=DS2A_DIR,
    verbose=True)

import pathlib, analyse_v2
analyse_v2.report(pathlib.Path(PROJECT_ROOT) / 'results' / 'ds2a_strong')

# did the search actually pick the strong members?
import json
for v in ['V4_enriched_pool', 'V7_strong_members']:
    sub = df2[df2.variant == v]
    if not len(sub):
        continue
    riem = sum('"RIEM"' in t for t in sub.topology)
    raw  = sum('"RAW"' in t for t in sub.topology)
    fb   = sum('"FB"' in t for t in sub.topology)
    print(f'{v}: tangent-space {riem}/{len(sub)}  exact-baseline/EEGNet {raw}/{len(sub)}  FBCSP {fb}/{len(sub)}')

## 7. What to conclude

Read the `W/T/L` column before the `Δ` column. On these sample sizes a difference of two or three points will usually come out as a tie, and a tie is the honest answer — it is what turned the original submission around.

If a variant shows a positive Δ **and** more significant wins than losses across 32 paired tests, that is a real result worth a follow-up paper. If everything ties, that is also a result: it says the ceiling on this pool and these cohorts is set by the data, not by the search — which is a cleaner and more useful claim than a marginal accuracy gain.